# BÀI THỰC HÀNH LECTURE 14: PHÂN TÍCH VÀ BIẾN ĐỔI DỮ LIỆU NÂNG CAO VỚI PYSPARK
## (Advanced PySpark: Aggregations, GroupBy, Window Functions & Joins)

**Khóa học**: Data Engineeringk    
**Tập dữ liệu sử dụng**: `online-retail-dataset.csv` (Dữ liệu giao dịch thương mại điện tử thực tế gồm 541,909 bản ghi)

---

### 🎯 Mục tiêu bài học (Learning Objectives):
1. **Làm sạch & Chuẩn bị dữ liệu quy mô lớn (Data Cleaning & Preparation)**:
   - Lọc bỏ các giao dịch hủy/trả hàng (`Quantity <= 0`, `UnitPrice <= 0`).
   - Xử lý các giá trị `null` ở định danh khách hàng (`CustomerID`).
   - Tính toán cột doanh thu nghiệp vụ `total_amount = Quantity * UnitPrice`.
   - Chuẩn hóa kiểu ngày tháng (`InvoiceDate` -> `DateType`).
2. **Làm chủ các hàm kết tập (Aggregation Functions)**:
   - Các phép tính thống kê cơ bản: `count`, `countDistinct`, `approx_count_distinct`, `sum`, `avg`, `min`, `max`.
   - Thống kê phương sai và độ lệch chuẩn: `stddev_sample`, `var_sample`.
   - Hệ số tương quan & hiệp phương sai: `corr`, `covar_samp`.
   - Kết tập dữ liệu dạng phức hợp (Complex Types): `collect_list`, `collect_set` (phục vụ phân tích giỏ hàng - Market Basket Analysis).
3. **Gom nhóm dữ liệu đa chiều (Grouping & Aggregation)**:
   - Kỹ thuật gom nhóm với `groupBy()`, kết hợp nhiều hàm tính toán qua `.agg()`.
   - Định danh bí danh (`alias`) cho các cột kết quả.
4. **Phân tích theo khung nhìn (Window Functions)**:
   - Hiểu sâu về cấu trúc Window Specification (`partitionBy`, `orderBy`, `rowsBetween`).
   - Phân biệt giữa `row_number()`, `rank()`, và `dense_rank()`.
   - Ứng dụng Window Functions để tìm Top N sản phẩm bán chạy nhất trong từng Quốc gia, tính tổng lũy kế (Cumulative Running Total).
5. **Tổng hợp nâng cao: Rollup, Cube & Pivot Table**:
   - `rollup`: Tổng hợp phân cấp đa tầng (Hierarchical Aggregation).
   - `cube`: Tổng hợp mọi tổ hợp chiều phân tích (OLAP Data Cube).
   - `pivot`: Xoay bảng từ hàng thành cột để tạo báo cáo ma trận doanh thu.
6. **Kỹ thuật kết nối dữ liệu (Joins in Apache Spark)**:
   - Thành thạo 8 kiểu Join: `inner`, `outer`, `left_outer`, `right_outer`, `left_semi`, `left_anti`, `cross`.
   - Xử lý xung đột trùng tên cột sau khi Join.
7. **Tối ưu hóa hiệu năng kết nối (Shuffle Join vs Broadcast Hash Join)**:
   - Phân biệt cơ chế vật lý giữa **Shuffle Hash Join** (Big Table join Big Table) và **Broadcast Hash Join** (Big Table join Small Table).
   - Tối ưu hóa với hàm `broadcast()` để triệt tiêu chi phí Shuffle qua mạng.
8. **Dự án thực hành tổng hợp (Hands-on Sales Analytics Projects)**: 5 bài toán thực tế kèm lời giải chi tiết.

---

### 🧭 Mục lục nội dung (Table of Contents):
- [0. Khởi tạo SparkSession & Thiết lập môi trường](#0-khởi-tạo-sparksession--thiết-lập-môi-trường)
- [1. Đọc dữ liệu & Làm sạch (Data Cleaning & Preparation)](#1-đọc-dữ-liệu--làm-sạch-data-cleaning--preparation)
- [2. Các hàm kết tập cơ bản & nâng cao (Aggregation Functions)](#2-các-hàm-kết-tập-cơ-bản--nâng-cao-aggregation-functions)
- [3. Gom nhóm dữ liệu với groupBy & agg](#3-gom-nhóm-dữ-liệu-với-groupby--agg)
- [4. Phân tích theo khung nhìn với Window Functions](#4-phân-tích-theo-khung-nhìn-với-window-functions)
- [5. Tổng hợp đa cấp: Rollup, Cube & Pivot Table](#5-tổng-hợp-đa-cấp-rollup-cube--pivot-table)
- [6. Kỹ thuật kết nối dữ liệu (Joins)](#6-kỹ-thuật-kết-nối-dữ-liệu-joins)
- [7. Tối ưu hóa hiệu năng: Shuffle Join vs Broadcast Join](#7-tối-ưu-hóa-hiệu-năng-shuffle-join-vs-broadcast-join)
- [8. Bài tập thực hành tổng hợp (Mini Projects)](#8-bài-tập-thực-hành-tổng-hợp-mini-projects)
- [9. Dọn dẹp tài nguyên & Giải phóng bộ nhớ](#9-dọn-dẹp-tài-nguyên--giải-phóng-bộ-nhớ)
- [10. Tổng kết & Điểm nối tiếp Buổi 15](#10-tổng-kết--điểm-nối-tiếp-buổi-15)

---
## 0. Khởi tạo SparkSession & Thiết lập môi trường

Trong bài học này, chúng ta sẽ phân tích tập dữ liệu thương mại điện tử `online-retail-dataset.csv` gồm hơn **541 nghìn giao dịch**.  
Đoạn mã sau sẽ cấu hình `SparkSession` với chế độ chạy đa luồng (`local[*]`) và thiết lập `spark.sql.shuffle.partitions = 5` để tối ưu hóa hiệu năng tính toán trên máy cục bộ.

In [ ]:
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, expr, lit, when, coalesce as spark_coalesce,
    count, countDistinct, approx_count_distinct,
    sum, avg, mean, min, max, first, last,
    stddev_samp, stddev_pop, var_samp, var_pop,
    corr, covar_samp,
    collect_list, collect_set,
    to_date, to_timestamp, date_format,
    dense_rank, rank, row_number,
    broadcast
)
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType, LongType
)
from pyspark.sql.window import Window
from pyspark.sql import Row

# 1. Tìm kiếm đường dẫn tệp dữ liệu online-retail-dataset.csv
candidate_paths = [
    "online-retail-dataset.csv",
    "lecture-14/online-retail-dataset.csv"
]
DATA_PATH = next((p for p in candidate_paths if os.path.exists(p)), "online-retail-dataset.csv")
print(f"[*] Đường dẫn dữ liệu được chọn: {DATA_PATH}")

# 2. Khởi tạo SparkSession
spark = SparkSession.builder \
    .appName("Lecture14_Lab_Aggregations_and_Joins") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "5") \
    .config("spark.driver.extraJavaOptions", "-Djdk.lang.Process.launchMechanism=vfork") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

print(f"[*] Apache Spark Version: {spark.version}")
print(f"[*] Spark Master: {spark.sparkContext.master}")
print(f"[*] Application Name: {spark.sparkContext.appName}")


[*] Đường dẫn dữ liệu được chọn: lecture-14/online-retail-dataset.csv
[*] Apache Spark Version: 4.2.0
[*] Spark Master: local[*]
[*] Application Name: Lecture14_Lab_Aggregations_and_Joins


---
## 1. Đọc dữ liệu & Làm sạch (Data Cleaning & Preparation)

### 💡 Bối cảnh dữ liệu Online Retail:
Tập dữ liệu chứa tất cả các giao dịch phát sinh từ ngày 01/12/2010 đến ngày 09/12/2011 của một công ty bán lẻ quà tặng trực tuyến có trụ sở tại Vương quốc Anh (UK).  
Các cột dữ liệu:
* `InvoiceNo`: Mã hóa đơn (chuỗi 6 chữ số, nếu bắt đầu bằng chữ 'C' nghĩa là đơn hàng bị hủy - Cancellation).
* `StockCode`: Mã định danh sản phẩm.
* `Description`: Tên sản phẩm.
* `Quantity`: Số lượng mua của từng sản phẩm trong giao dịch (âm nếu là hàng trả lại).
* `InvoiceDate`: Ngày và giờ phát sinh hóa đơn.
* `UnitPrice`: Đơn giá mỗi sản phẩm (bằng Bảng Anh - GBP).
* `CustomerID`: Mã khách hàng (nhiều bản ghi có giá trị null do khách vãng lai).
* `Country`: Quốc gia nơi khách hàng cư trú.

In [2]:
# Đọc tệp CSV với header và tự suy luận kiểu dữ liệu
raw_df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(DATA_PATH)

total_raw_rows = raw_df.count()
print(f"[*] Tổng số bản ghi thô: {total_raw_rows:,} dòng")
raw_df.printSchema()
raw_df.show(5, truncate=False)

[*] Tổng số bản ghi thô: 541,909 dòng
root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: string (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- Country: string (nullable = true)

+---------+---------+-----------------------------------+--------+--------------+---------+----------+--------------+
|InvoiceNo|StockCode|Description                        |Quantity|InvoiceDate   |UnitPrice|CustomerID|Country       |
+---------+---------+-----------------------------------+--------+--------------+---------+----------+--------------+
|536365   |85123A   |WHITE HANGING HEART T-LIGHT HOLDER |6       |12/1/2010 8:26|2.55     |17850     |United Kingdom|
|536365   |71053    |WHITE METAL LANTERN                |6       |12/1/2010 8:26|3.39     |17850     |United Kingdom|
|536365   |84406B   |CREAM CUPID H

### Chuẩn hóa & Làm sạch dữ liệu (Data Cleaning Pipeline):
Trong thực tế Data Engineering, dữ liệu thô luôn chứa các bản ghi dị thường:
1. Đơn hàng trả lại hoặc bị lỗi có `Quantity <= 0`.
2. Đơn giá bị sai lệch hoặc mẫu dùng thử có `UnitPrice <= 0`.
3. Cột `InvoiceDate` đang ở dạng chuỗi, cần chuyển sang định dạng ngày tháng (`DateType` / `TimestampType`).
4. Thêm cột tính toán doanh thu dòng: `total_amount = Quantity * UnitPrice`.
5. Đưa dữ liệu sạch vào bộ nhớ đệm (`cache()`) để tăng tốc các phép toán tổng hợp tiếp theo.

In [3]:
# Làm sạch dữ liệu và tạo thêm các trường phục vụ phân tích
clean_df = raw_df \
    .filter((col("Quantity") > 0) & (col("UnitPrice") > 0)) \
    .withColumn("total_amount", col("Quantity") * col("UnitPrice")) \
    .withColumn("InvoiceTimestamp", to_timestamp(col("InvoiceDate"), "M/d/yyyy H:m")) \
    .withColumn("date", to_date(col("InvoiceTimestamp"))) \
    .coalesce(5)

# Lưu vào cache để tái sử dụng nhanh chóng
clean_df.cache()
clean_count = clean_df.count()
print(f"[*] Số bản ghi sau khi làm sạch: {clean_count:,} dòng (Loại bỏ {total_raw_rows - clean_count:,} dòng lỗi)")

# Đăng ký View tạm thời phục vụ Spark SQL
clean_df.createOrReplaceTempView("online_retail")
clean_df.select("InvoiceNo", "StockCode", "Description", "Quantity", "UnitPrice", "total_amount", "Country", "date").show(5)

[*] Số bản ghi sau khi làm sạch: 530,104 dòng (Loại bỏ 11,805 dòng lỗi)
+---------+---------+--------------------+--------+---------+------------------+--------------+----------+
|InvoiceNo|StockCode|         Description|Quantity|UnitPrice|      total_amount|       Country|      date|
+---------+---------+--------------------+--------+---------+------------------+--------------+----------+
|   536365|   85123A|WHITE HANGING HEA...|       6|     2.55|15.299999999999999|United Kingdom|2010-12-01|
|   536365|    71053| WHITE METAL LANTERN|       6|     3.39|             20.34|United Kingdom|2010-12-01|
|   536365|   84406B|CREAM CUPID HEART...|       8|     2.75|              22.0|United Kingdom|2010-12-01|
|   536365|   84029G|KNITTED UNION FLA...|       6|     3.39|             20.34|United Kingdom|2010-12-01|
|   536365|   84029E|RED WOOLLY HOTTIE...|       6|     3.39|             20.34|United Kingdom|2010-12-01|
+---------+---------+--------------------+--------+---------+-----------

---
## 2. Các hàm kết tập cơ bản & nâng cao (Aggregation Functions)

Kết tập (Aggregation) là thao tác cốt lõi trong xử lý dữ liệu lớn. Các hàm này nhận vào nhiều dòng dữ liệu và tóm tắt thành một giá trị đại diện.

### 2.1. Đếm số lượng: count, countDistinct, approx_count_distinct
* `count("col")`: Đếm số lượng bản ghi không null trong cột.
* `countDistinct("col")`: Đếm chính xác số lượng giá trị duy nhất (đòi hỏi shuffle dữ liệu).
* `approx_count_distinct("col", rsd)`: Đếm **xấp xỉ** số lượng giá trị duy nhất với sai số tương đối (`rsd` - Relative Standard Deviation, mặc định là 0.05 tức sai số khoảng 5%). Trên tập dữ liệu hàng trăm triệu dòng, hàm xấp xỉ này chạy nhanh gấp hàng chục lần so với đếm chính xác mà vẫn đảm bảo độ tin cậy kinh doanh!

In [4]:
clean_df.select(
    count("InvoiceNo").alias("total_rows"),
    countDistinct("CustomerID").alias("exact_unique_customers"),
    approx_count_distinct("CustomerID", 0.05).alias("approx_unique_customers"),
    countDistinct("StockCode").alias("unique_products")
).show()

+----------+----------------------+-----------------------+---------------+
|total_rows|exact_unique_customers|approx_unique_customers|unique_products|
+----------+----------------------+-----------------------+---------------+
|    530104|                  4338|                   4450|           3922|
+----------+----------------------+-----------------------+---------------+



### 2.2. Min, Max, Sum, Avg (Mean)
Các hàm đo lường độ lớn và xu hướng tập trung của dữ liệu số.

In [5]:
clean_df.select(
    min("Quantity").alias("min_qty"),
    max("Quantity").alias("max_qty"),
    min("UnitPrice").alias("min_price"),
    max("UnitPrice").alias("max_price"),
    sum("Quantity").alias("total_items_sold"),
    sum("total_amount").alias("total_revenue"),
    avg("total_amount").alias("avg_line_revenue")
).show()

+-------+-------+---------+---------+----------------+--------------------+-----------------+
|min_qty|max_qty|min_price|max_price|total_items_sold|       total_revenue| avg_line_revenue|
+-------+-------+---------+---------+----------------+--------------------+-----------------+
|      1|  80995|    0.001| 13541.33|         5588376|1.0666684543997977E7|20.12187145163586|
+-------+-------+---------+---------+----------------+--------------------+-----------------+



### 2.3. Phương sai, Độ lệch chuẩn & Tương quan (Variance, StdDev, Correlation)
* `stddev_samp` / `stddev_pop`: Độ lệch chuẩn mẫu / tổng thể.
* `corr("colA", "colB")`: Hệ số tương quan Pearson giữa hai biến (giá trị từ -1 đến 1).

In [6]:
clean_df.select(
    stddev_samp("Quantity").alias("qty_stddev"),
    var_samp("Quantity").alias("qty_variance"),
    corr("Quantity", "UnitPrice").alias("qty_price_correlation")
).show()

+------------------+-----------------+---------------------+
|        qty_stddev|     qty_variance|qty_price_correlation|
+------------------+-----------------+---------------------+
|155.52412351063683|24187.75299375182| -0.00377254266163...|
+------------------+-----------------+---------------------+



### 2.4. Kết tập thành kiểu dữ liệu phức hợp: collect_list và collect_set
* `collect_list("col")`: Gom tất cả các giá trị của một nhóm vào một mảng Python (Array), giữ nguyên thứ tự và giá trị lặp lại.
* `collect_set("col")`: Gom tất cả các giá trị vào một tập hợp (Set), **tự động loại bỏ trùng lặp**.

> [!TIP]
> **ỨNG DỤNG THỰC TẾ (Market Basket Analysis - Khai phá giỏ hàng):**  
> `collect_set` là công cụ cực kỳ hữu ích để gom toàn bộ các mã sản phẩm (`StockCode`) mua trong cùng một hóa đơn (`InvoiceNo`) để phân tích xem những sản phẩm nào thường được khách hàng mua kèm với nhau!

In [7]:
# Gom danh sách các sản phẩm duy nhất trong 3 hóa đơn đầu tiên
basket_sample = clean_df.limit(20) \
    .groupBy("InvoiceNo") \
    .agg(
        collect_list("StockCode").alias("items_list"),
        collect_set("StockCode").alias("items_set")
    )

basket_sample.show(3, truncate=False)

+---------+-----------------------------------------------------------------------------+-----------------------------------------------------------------------------+
|InvoiceNo|items_list                                                                   |items_set                                                                    |
+---------+-----------------------------------------------------------------------------+-----------------------------------------------------------------------------+
|536365   |[85123A, 71053, 84406B, 84029G, 84029E, 22752, 21730]                        |[71053, 84029G, 84406B, 22752, 21730, 84029E, 85123A]                        |
|536366   |[22633, 22632]                                                               |[22632, 22633]                                                               |
|536367   |[84879, 22745, 22748, 22749, 22310, 84969, 22623, 22622, 21754, 21755, 21777]|[22745, 21777, 22748, 22749, 22310, 22622, 21755, 84969, 22623, 21754, 

---
## 3. Gom nhóm dữ liệu với groupBy & agg

Gom nhóm (`groupBy`) cho phép bạn phân chia dữ liệu thành các nhóm theo một hoặc nhiều cột khóa, sau đó áp dụng một hoặc nhiều hàm tổng hợp trên từng nhóm.

### 3.1. Gom nhóm cơ bản với count()

In [8]:
# Đếm số lượng dòng giao dịch theo từng Quốc gia (Top 5 quốc gia hàng đầu)
clean_df.groupBy("Country").count().orderBy(col("count").desc()).show(5)

+--------------+------+
|       Country| count|
+--------------+------+
|United Kingdom|485123|
|       Germany|  9040|
|        France|  8407|
|          EIRE|  7890|
|         Spain|  2484|
+--------------+------+
only showing top 5 rows


### 3.2. Gom nhóm nâng cao với .agg()
Phương thức `.agg()` cho phép bạn áp dụng đồng thời nhiều hàm tổng hợp khác nhau trên nhiều cột, đồng thời đặt bí danh (`alias`) chuyên nghiệp.

In [9]:
# Thống kê hiệu suất kinh doanh toàn diện theo Quốc gia
country_summary = clean_df.groupBy("Country").agg(
    countDistinct("InvoiceNo").alias("total_orders"),
    countDistinct("CustomerID").alias("unique_customers"),
    sum("Quantity").alias("total_units_sold"),
    sum("total_amount").alias("total_revenue"),
    avg("total_amount").alias("avg_revenue_per_line")
).orderBy(col("total_revenue").desc())

print("Top 10 thị trường có doanh thu lớn nhất:")
country_summary.show(10)

Top 10 thị trường có doanh thu lớn nhất:
+--------------+------------+----------------+----------------+------------------+--------------------+
|       Country|total_orders|unique_customers|total_units_sold|     total_revenue|avg_revenue_per_line|
+--------------+------------+----------------+----------------+------------------+--------------------+
|United Kingdom|       18019|            3920|         4662390| 9025222.083998322|  18.603987203241903|
|   Netherlands|          94|               9|          200361|285446.33999999997|   121.0031114879186|
|          EIRE|         288|               3|          147173|283453.95999999973|   35.92572370088716|
|       Germany|         457|              94|          119261|228867.13999999996|  25.317161504424774|
|        France|         392|              87|          112103|209715.11000000025|  24.945296776495805|
|     Australia|          57|               9|           83901|         138521.31|  117.19230964467005|
|         Spain|       

### 3.3. Gom nhóm bằng từ điển (Dictionary Mapping)
Spark cũng hỗ trợ cú pháp truyền từ điển `{column_name: aggregation_function}` vào `.agg()` để viết ngắn gọn.

In [10]:
# Gom nhóm sử dụng Dictionary mapping
clean_df.groupBy("Country").agg({
    "total_amount": "sum",
    "Quantity": "avg"
}).orderBy(col("sum(total_amount)").desc()).show(5)

+--------------+------------------+------------------+
|       Country| sum(total_amount)|     avg(Quantity)|
+--------------+------------------+------------------+
|United Kingdom| 9025222.083998322| 9.610737895337884|
|   Netherlands|285446.33999999997| 84.93471810089021|
|          EIRE|283453.95999999973|18.653105196451204|
|       Germany|228867.13999999996| 13.19258849557522|
|        France|209715.11000000025|13.334483168787914|
+--------------+------------------+------------------+
only showing top 5 rows


---
## 4. Phân tích theo khung nhìn với Window Functions

> [!IMPORTANT]
> **Khái niệm Window Function:**  
> Trong khi `groupBy` gom nhiều dòng thành một dòng duy nhất đại diện cho nhóm, thì **Window Function tính toán kết quả trên một nhóm các dòng (gọi là Window Frame) nhưng VẪN GIỮ NGUYÊN từng dòng dữ liệu ban đầu**.

### Cấu trúc một Window Specification (`Window.partitionBy(...).orderBy(...)`):
1. `partitionBy()`: Chia dữ liệu thành các phân vùng logic (ví dụ theo từng Khách hàng hoặc từng Quốc gia).
2. `orderBy()`: Xác định thứ tự các dòng trong từng phân vùng.
3. `rowsBetween()`: (Tùy chọn) Xác định khung nhìn các dòng liên quan (ví dụ từ đầu phân vùng đến dòng hiện tại).

### Phân biệt 3 hàm xếp hạng quan trọng:
* **`row_number()`**: Đánh số thứ tự tuần tự không trùng lặp (`1, 2, 3, 4...`).
* **`rank()`**: Xếp hạng có nhảy bậc khi có điểm số bằng nhau (`1, 2, 2, 4...`).
* **`dense_rank()`**: Xếp hạng liên tục, KHÔNG nhảy bậc khi có điểm số bằng nhau (`1, 2, 2, 3...`).

In [11]:
# 1. Định nghĩa Window Specification: Phân vùng theo Khách hàng và sắp xếp giảm dần theo doanh thu dòng
customer_window = Window \
    .partitionBy("CustomerID") \
    .orderBy(col("total_amount").desc())

# 2. Xếp hạng các giao dịch có giá trị lớn nhất của từng khách hàng
ranked_customer_purchases = clean_df \
    .filter(col("CustomerID").isNotNull()) \
    .select(
        col("CustomerID"),
        col("InvoiceNo"),
        col("Description"),
        col("total_amount"),
        row_number().over(customer_window).alias("row_num"),
        rank().over(customer_window).alias("rank"),
        dense_rank().over(customer_window).alias("dense_rank")
    )

print("Top giao dịch của khách hàng với các hàm xếp hạng:")
ranked_customer_purchases.filter(col("CustomerID") == 17850).show(6, truncate=False)

Top giao dịch của khách hàng với các hàm xếp hạng:
+----------+---------+------------------------+------------+-------+----+----------+
|CustomerID|InvoiceNo|Description             |total_amount|row_num|rank|dense_rank|
+----------+---------+------------------------+------------+-------+----+----------+
|17850     |536790   |IVORY EMBROIDERED QUILT |107.25      |1      |1   |1         |
|17850     |536396   |IVORY EMBROIDERED QUILT |71.5        |2      |2   |2         |
|17850     |536406   |IVORY EMBROIDERED QUILT |71.5        |3      |2   |2         |
|17850     |536600   |IVORY EMBROIDERED QUILT |71.5        |4      |2   |2         |
|17850     |536685   |IVORY EMBROIDERED QUILT |71.5        |5      |2   |2         |
|17850     |536690   |IVORY EMBROIDERED QUILT |71.5        |6      |2   |2         |
+----------+---------+------------------------+------------+-------+----+----------+
only showing top 6 rows


### Ứng dụng thực tế: Tính tổng doanh thu lũy kế (Cumulative Running Total)
Sử dụng `Window` với khung nhìn `rowsBetween(Window.unboundedPreceding, Window.currentRow)` để tính tổng lũy kế theo thời gian.

In [12]:
# Tính doanh thu theo ngày
daily_sales = clean_df.groupBy("date").agg(sum("total_amount").alias("daily_revenue")).orderBy("date")

# Định nghĩa Window tính tổng lũy kế từ ngày đầu tiên đến ngày hiện tại
running_total_window = Window \
    .orderBy("date") \
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)

daily_running_total = daily_sales.withColumn(
    "cumulative_revenue",
    sum("daily_revenue").over(running_total_window)
)

print("Doanh thu theo ngày và tổng tích lũy theo thời gian:")
daily_running_total.show(7)

Doanh thu theo ngày và tổng tích lũy theo thời gian:
+----------+------------------+------------------+
|      date|     daily_revenue|cumulative_revenue|
+----------+------------------+------------------+
|2010-12-01| 58960.79000000025| 58960.79000000025|
|2010-12-02| 47748.37999999988|106709.17000000013|
|2010-12-03| 46943.70999999988|         153652.88|
|2010-12-05|31774.950000000164|185427.83000000016|
|2010-12-06|54830.460000000014|240258.29000000018|
|2010-12-07| 99618.20000000104| 339876.4900000012|
|2010-12-08|45389.979999999814|  385266.470000001|
+----------+------------------+------------------+
only showing top 7 rows


---
## 5. Tổng hợp đa cấp: Rollup, Cube & Pivot Table

Khi xây dựng các báo cáo phân tích kinh doanh (BI & OLAP), ta thường phải tổng hợp dữ liệu ở nhiều cấp độ phân cấp khác nhau.

### 5.1. Rollup (Tổng hợp phân cấp từ chi tiết đến toàn cục)
`rollup("A", "B")` sẽ tự động tính:
1. Tổng hợp theo từng cặp `(A, B)`.
2. Tổng hợp theo từng nhóm `A` (với `B` mang giá trị `null`).
3. Tổng hợp toàn cục toàn bộ dữ liệu (với cả `A` và `B` mang giá trị `null`).

In [13]:
# Lọc tập mẫu gồm 2 quốc gia: France và Germany để dễ quan sát kết quả
sample_countries_df = clean_df.filter(col("Country").isin("France", "Germany"))

rolled_up_df = sample_countries_df \
    .rollup("Country", "date") \
    .agg(sum("total_amount").alias("revenue")) \
    .orderBy("Country", "date")

print("Kết quả Rollup (Chú ý các dòng có giá trị null đại diện cho tổng mức cha):")
rolled_up_df.show(10)

Kết quả Rollup (Chú ý các dòng có giá trị null đại diện cho tổng mức cha):
+-------+----------+------------------+
|Country|      date|           revenue|
+-------+----------+------------------+
|   NULL|      NULL| 438582.2500000005|
| France|      NULL|209715.11000000025|
| France|2010-12-01|            855.86|
| France|2010-12-03|            425.38|
| France|2010-12-05|           1526.92|
| France|2010-12-07|           1448.98|
| France|2010-12-08|            173.55|
| France|2010-12-09|           1859.73|
| France|2010-12-10|            1071.9|
| France|2010-12-15|            195.89|
+-------+----------+------------------+
only showing top 10 rows


### 5.2. Cube (Tổng hợp toàn bộ các tổ hợp chiều phân tích)
`cube("A", "B")` tính toán tổng hợp cho **tất cả các tổ hợp có thể có**: `(A, B)`, `(A, null)`, `(null, B)`, và `(null, null)`.

In [14]:
cubed_df = sample_countries_df \
    .cube("Country", "StockCode") \
    .agg(sum("Quantity").alias("total_quantity")) \
    .orderBy("Country", "StockCode")

print("Kết quả Cube trên Country và StockCode:")
cubed_df.show(5)

Kết quả Cube trên Country và StockCode:
+-------+---------+--------------+
|Country|StockCode|total_quantity|
+-------+---------+--------------+
|   NULL|     NULL|        231364|
|   NULL|    10002|           373|
|   NULL|    10120|            10|
|   NULL|    10125|           590|
|   NULL|    10135|           210|
+-------+---------+--------------+
only showing top 5 rows


### 5.3. Pivot Table (Xoay bảng ma trận)
Xoay các giá trị riêng biệt của một cột thành các cột mới trong DataFrame để tạo báo cáo so sánh đa chiều.

In [15]:
# Tạo bảng ma trận doanh thu của các nước Châu Âu lớn theo tháng
monthly_df = clean_df \
    .filter(col("Country").isin("France", "Germany", "Spain")) \
    .withColumn("month", date_format(col("date"), "yyyy-MM"))

pivot_table = monthly_df \
    .groupBy("month") \
    .pivot("Country") \
    .agg(sum("total_amount")) \
    .orderBy("month")

print("Báo cáo ma trận doanh thu theo tháng giữa France, Germany và Spain:")
pivot_table.show(6)

Báo cáo ma trận doanh thu theo tháng giữa France, Germany và Spain:
+-------+------------------+------------------+------------------+
|  month|            France|           Germany|             Spain|
+-------+------------------+------------------+------------------+
|2010-12|  9616.30999999999|15241.139999999996|1843.7299999999998|
|2011-01| 17740.12000000001|16910.840000000015|10086.090000000013|
|2011-02| 8515.959999999997| 9581.049999999996|2114.5000000000014|
|2011-03|          14589.55|14392.689999999986|           5363.15|
|2011-04| 5529.609999999997|12315.539999999999|1785.6500000000008|
|2011-05|17614.879999999994|25751.199999999997|            3257.6|
+-------+------------------+------------------+------------------+
only showing top 6 rows


---
## 6. Kỹ thuật kết nối dữ liệu (Joins in Apache Spark)

Trong một kiến trúc dữ liệu doanh nghiệp, dữ liệu bán hàng (`Fact Table`) thường xuyên phải được kết hợp với các bảng thông tin mô tả (`Dimension Tables`) như Khách hàng, Sản phẩm, Địa lý.

### 💡 8 Kiểu Join trong Apache Spark:
1. **`inner`**: Chỉ giữ các dòng có khóa xuất hiện ở cả hai bảng.
2. **`outer` / `full_outer`**: Giữ lại toàn bộ dòng của cả hai bảng; điền `null` vào các vị trí không khớp.
3. **`left_outer` / `left`**: Giữ toàn bộ dòng của bảng bên trái; điền `null` nếu bảng phải không khớp.
4. **`right_outer` / `right`**: Giữ toàn bộ dòng của bảng bên phải.
5. **`left_semi`**: Giữ các dòng của bảng bên trái có khóa tồn tại ở bảng bên phải (nhưng **không lấy cột** của bảng phải, tương đương `EXISTS`).
6. **`left_anti`**: Giữ các dòng của bảng bên trái có khóa **KHÔNG** tồn tại ở bảng bên phải (tương đương `NOT EXISTS`).
7. **`cross`**: Tích Descartes (ghép mọi dòng bảng trái với mọi dòng bảng phải - cực kỳ nguy hiểm trên Big Data).

Để nắm vững bản chất, chúng ta sẽ tạo hai tập dữ liệu mô phỏng nhỏ trước khi áp dụng trên tập dữ liệu lớn.

In [16]:
# Khởi tạo 2 bảng dữ liệu mô phỏng để minh họa rõ các trường hợp Join
customers_data = [
    (1, "Alice", "Hanoi", "VIP"),
    (2, "Bob", "Danang", "Gold"),
    (3, "Charlie", "Saigon", "Silver"),
    (4, "David", "Cantho", "Standard"),
    (5, "Emma", "Haiphong", "Standard")  # Khách hàng Emma chưa có đơn hàng nào
]
customers_df = spark.createDataFrame(
    customers_data, 
    ["customer_id", "customer_name", "city", "tier"]
)

orders_data = [
    (101, 1, "2023-01-10", 500.0),
    (102, 1, "2023-01-15", 300.0),
    (103, 2, "2023-01-16", 150.0),
    (104, 3, "2023-01-20", 700.0),
    (105, 99, "2023-01-22", 250.0)  # Đơn hàng của khách 99 không có trong bảng customers (dị thường)
]
orders_df = spark.createDataFrame(
    orders_data, 
    ["order_id", "customer_id", "order_date", "amount"]
)

print("=== Bảng Khách hàng (Customers): ===")
customers_df.show()
print("=== Bảng Đơn hàng (Orders): ===")
orders_df.show()

=== Bảng Khách hàng (Customers): ===
+-----------+-------------+--------+--------+
|customer_id|customer_name|    city|    tier|
+-----------+-------------+--------+--------+
|          1|        Alice|   Hanoi|     VIP|
|          2|          Bob|  Danang|    Gold|
|          3|      Charlie|  Saigon|  Silver|
|          4|        David|  Cantho|Standard|
|          5|         Emma|Haiphong|Standard|
+-----------+-------------+--------+--------+

=== Bảng Đơn hàng (Orders): ===
+--------+-----------+----------+------+
|order_id|customer_id|order_date|amount|
+--------+-----------+----------+------+
|     101|          1|2023-01-10| 500.0|
|     102|          1|2023-01-15| 300.0|
|     103|          2|2023-01-16| 150.0|
|     104|          3|2023-01-20| 700.0|
|     105|         99|2023-01-22| 250.0|
+--------+-----------+----------+------+



In [17]:
# 1. INNER JOIN: Chỉ lấy đơn hàng có thông tin khách hàng hợp lệ
print("--- 1. INNER JOIN ---")
customers_df.join(orders_df, on="customer_id", how="inner").show()

# 2. LEFT JOIN: Giữ toàn bộ khách hàng, kể cả khách hàng chưa mua đơn nào (Emma)
print("--- 2. LEFT JOIN ---")
customers_df.join(orders_df, on="customer_id", how="left").show()

# 3. RIGHT JOIN: Giữ toàn bộ đơn hàng, kể cả đơn hàng của khách lạ (ID 99)
print("--- 3. RIGHT JOIN ---")
customers_df.join(orders_df, on="customer_id", how="right").show()

# 4. FULL OUTER JOIN: Giữ toàn bộ bản ghi từ cả 2 bảng
print("--- 4. FULL OUTER JOIN ---")
customers_df.join(orders_df, on="customer_id", how="outer").show()

# 5. LEFT SEMI JOIN: Tìm các khách hàng ĐÃ TỪNG MUA HÀNG (chỉ lấy cột của customers)
print("--- 5. LEFT SEMI JOIN ---")
customers_df.join(orders_df, on="customer_id", how="left_semi").show()

# 6. LEFT ANTI JOIN: Tìm các khách hàng CHƯA TỪNG MUA ĐƠN NÀO (phát hiện đối tượng cần remarketing)
print("--- 6. LEFT ANTI JOIN (Khách hàng chưa mua hàng) ---")
customers_df.join(orders_df, on="customer_id", how="left_anti").show()

--- 1. INNER JOIN ---
+-----------+-------------+------+------+--------+----------+------+
|customer_id|customer_name|  city|  tier|order_id|order_date|amount|
+-----------+-------------+------+------+--------+----------+------+
|          1|        Alice| Hanoi|   VIP|     102|2023-01-15| 300.0|
|          1|        Alice| Hanoi|   VIP|     101|2023-01-10| 500.0|
|          2|          Bob|Danang|  Gold|     103|2023-01-16| 150.0|
|          3|      Charlie|Saigon|Silver|     104|2023-01-20| 700.0|
+-----------+-------------+------+------+--------+----------+------+

--- 2. LEFT JOIN ---
+-----------+-------------+--------+--------+--------+----------+------+
|customer_id|customer_name|    city|    tier|order_id|order_date|amount|
+-----------+-------------+--------+--------+--------+----------+------+
|          1|        Alice|   Hanoi|     VIP|     102|2023-01-15| 300.0|
|          1|        Alice|   Hanoi|     VIP|     101|2023-01-10| 500.0|
|          2|          Bob|  Danang|   

### Xử lý vấn đề trùng lặp tên cột sau khi Join
Khi hai bảng có cùng tên cột khóa (ví dụ `customer_id`), nếu join bằng biểu thức điều kiện `df1.id == df2.id`, DataFrame kết quả sẽ có hai cột cùng tên, gây lỗi khi gọi `select("customer_id")`.

**3 Cách xử lý chuẩn:**
1. **Truyền chuỗi tên cột khóa**: `df1.join(df2, on="customer_id")` (Spark tự động gộp thành 1 cột duy nhất).
2. **Đổi tên cột trước khi join**: Dùng `withColumnRenamed`.
3. **Xóa cột thừa sau khi join**: `df1.join(df2, condition).drop(df2["customer_id"])`.

In [18]:
# Minh họa cách xử lý tốt nhất: Truyền tên cột dạng chuỗi
joined_clean = customers_df.join(orders_df, on="customer_id", how="inner")
print("Danh sách cột sau khi join bằng tên cột khóa (Không bị trùng lặp):", joined_clean.columns)
joined_clean.select("customer_id", "customer_name", "amount").show(2)

Danh sách cột sau khi join bằng tên cột khóa (Không bị trùng lặp): ['customer_id', 'customer_name', 'city', 'tier', 'order_id', 'order_date', 'amount']
+-----------+-------------+------+
|customer_id|customer_name|amount|
+-----------+-------------+------+
|          1|        Alice| 500.0|
|          1|        Alice| 300.0|
+-----------+-------------+------+
only showing top 2 rows


---
## 7. Tối ưu hóa hiệu năng: Shuffle Join vs Broadcast Join

> [!CAUTION]
> **Nút thắt cổ chai lớn nhất trong Big Data: SHUFFLE**  
> Khi thực hiện Join giữa hai bảng dữ liệu lớn, Spark phải băm dữ liệu theo khóa (Hash Partitioning) và truyền tải dữ liệu qua lại giữa tất cả các máy trong cụm (All-to-All Network Communication).  
> Quá trình này tiêu tốn rất nhiều I/O mạng, đĩa và CPU, thường là nguyên nhân chính gây chậm hoặc sập Job.

### Giải pháp tối ưu: BROADCAST HASH JOIN (Map-Side Join)
* Khi một trong hai bảng là **bảng nhỏ (dưới 10MB mặc định, hoặc bảng danh mục lookup)**, ta có thể yêu cầu Spark **sao chép toàn bộ bảng nhỏ này tới bộ nhớ của tất cả các Worker Nodes**.
* Khi đó, mỗi Task chỉ cần đọc phần dữ liệu bảng lớn trên node của mình và tra cứu tại chỗ trên bảng nhỏ trong RAM -> **HOÀN TOÀN KHÔNG CẦN SHUFFLE QUA MẠNG!**
* Cú pháp: `large_df.join(broadcast(small_df), on="...")`.

In [19]:
# Tạo một bảng nhỏ tra cứu thông tin khu vực địa lý của các Quốc gia
country_region_data = [
    ("United Kingdom", "Western Europe", "GBP"),
    ("Germany", "Central Europe", "EUR"),
    ("France", "Western Europe", "EUR"),
    ("Spain", "Southern Europe", "EUR"),
    ("Australia", "Oceania", "AUD"),
    ("Japan", "East Asia", "JPY")
]
country_region_df = spark.createDataFrame(
    country_region_data, 
    ["Country", "Region", "Currency"]
)

# Thực hiện Broadcast Hash Join giữa bảng bán hàng 541 nghìn dòng và bảng lookup nhỏ
broadcast_joined_df = clean_df.join(
    broadcast(country_region_df), 
    on="Country", 
    how="left"
)

broadcast_joined_df.select("InvoiceNo", "Country", "Region", "Currency", "total_amount").show(5)

+---------+--------------+--------------+--------+------------------+
|InvoiceNo|       Country|        Region|Currency|      total_amount|
+---------+--------------+--------------+--------+------------------+
|   536365|United Kingdom|Western Europe|     GBP|15.299999999999999|
|   536365|United Kingdom|Western Europe|     GBP|             20.34|
|   536365|United Kingdom|Western Europe|     GBP|              22.0|
|   536365|United Kingdom|Western Europe|     GBP|             20.34|
|   536365|United Kingdom|Western Europe|     GBP|             20.34|
+---------+--------------+--------------+--------+------------------+
only showing top 5 rows


In [20]:
# Kiểm tra Kế hoạch thực thi (Execution Plan) để thấy BroadcastHashJoin
print("=== Execution Plan của Broadcast Join: ===")
broadcast_joined_df.explain()

=== Execution Plan của Broadcast Join: ===
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [Country#24, InvoiceNo#17, StockCode#18, Description#19, Quantity#20, InvoiceDate#21, UnitPrice#22, CustomerID#23, total_amount#71, InvoiceTimestamp#72, date#73, Region#8032, Currency#8033]
   +- BroadcastHashJoin [Country#24], [Country#8031], LeftOuter, BuildRight, false, false
      :- InMemoryTableScan [InvoiceNo#17, StockCode#18, Description#19, Quantity#20, InvoiceDate#21, UnitPrice#22, CustomerID#23, Country#24, total_amount#71, InvoiceTimestamp#72, date#73]
      :     +- InMemoryRelation [InvoiceNo#17, StockCode#18, Description#19, Quantity#20, InvoiceDate#21, UnitPrice#22, CustomerID#23, Country#24, total_amount#71, InvoiceTimestamp#72, date#73], StorageLevel(disk, memory, deserialized, 1 replicas)
      :           +- Coalesce 5
      :              +- *(1) Project [InvoiceNo#17, StockCode#18, Description#19, Quantity#20, InvoiceDate#21, UnitPrice#22, CustomerID#23, C

---
## 8. Bài tập thực hành tổng hợp (Mini Projects)

Hãy vận dụng toàn bộ kiến thức về Aggregation, Window Functions và Joins để giải quyết 5 bài toán thực tế dưới đây.

---

### 📝 Bài tập 1: Phân tích khách hàng giá trị cao (RFM Foundation)
**Yêu cầu**:
1. Từ `clean_df`, lọc các giao dịch có `CustomerID` hợp lệ (không null).
2. Gom nhóm theo `CustomerID`.
3. Tính toán 3 chỉ số cho mỗi khách hàng:
   - `order_count`: Số lượng hóa đơn khác nhau (`countDistinct("InvoiceNo")`).
   - `total_items`: Tổng số lượng sản phẩm đã mua (`sum("Quantity")`).
   - `total_spent`: Tổng số tiền đã chi trả (`sum("total_amount")`).
4. Sắp xếp giảm dần theo `total_spent` và in ra Top 5 khách hàng VIP nhất.

In [21]:
# --- LỜI GIẢI BÀI TẬP 1 ---
solution_1 = clean_df \
    .filter(col("CustomerID").isNotNull()) \
    .groupBy("CustomerID") \
    .agg(
        countDistinct("InvoiceNo").alias("order_count"),
        sum("Quantity").alias("total_items"),
        sum("total_amount").alias("total_spent")
    ) \
    .orderBy(col("total_spent").desc())

print("Top 5 khách hàng có tổng chi tiêu cao nhất:")
solution_1.show(5)

Top 5 khách hàng có tổng chi tiêu cao nhất:
+----------+-----------+-----------+------------------+
|CustomerID|order_count|total_items|       total_spent|
+----------+-----------+-----------+------------------+
|     14646|         73|     196915|280206.01999999996|
|     18102|         60|      64124|259657.30000000002|
|     17450|         46|      69993|194550.78999999992|
|     16446|          2|      80997|          168472.5|
|     14911|        201|      80265|143825.06000000008|
+----------+-----------+-----------+------------------+
only showing top 5 rows


---
### 📝 Bài tập 2: Top 3 sản phẩm bán chạy nhất trong từng Quốc gia (Window Function)
**Yêu cầu**:
1. Lọc dữ liệu trong 3 thị trường: `"Germany"`, `"France"`, `"Spain"`.
2. Gom nhóm theo `Country`, `StockCode`, và `Description` để tính tổng số lượng bán ra (`sum("Quantity")`).
3. Sử dụng `Window.partitionBy("Country").orderBy(col("total_units").desc())` kết hợp `dense_rank()`.
4. Lọc lấy đúng Top 3 sản phẩm bán chạy nhất cho mỗi quốc gia.

In [22]:
# --- LỜI GIẢI BÀI TẬP 2 ---
target_countries = ["Germany", "France", "Spain"]

product_sales_per_country = clean_df \
    .filter(col("Country").isin(target_countries)) \
    .groupBy("Country", "StockCode", "Description") \
    .agg(sum("Quantity").alias("total_units"))

country_product_window = Window \
    .partitionBy("Country") \
    .orderBy(col("total_units").desc())

top_3_per_country = product_sales_per_country \
    .withColumn("rank", dense_rank().over(country_product_window)) \
    .filter(col("rank") <= 3) \
    .orderBy("Country", "rank")

print("Top 3 sản phẩm bán chạy nhất theo từng quốc gia:")
top_3_per_country.show(truncate=False)

Top 3 sản phẩm bán chạy nhất theo từng quốc gia:
+-------+---------+-----------------------------------+-----------+----+
|Country|StockCode|Description                        |total_units|rank|
+-------+---------+-----------------------------------+-----------+----+
|France |23084    |RABBIT NIGHT LIGHT                 |4024       |1   |
|France |22492    |MINI PAINT SET VINTAGE             |2196       |2   |
|France |21731    |RED TOADSTOOL LED NIGHT LIGHT      |1315       |3   |
|Germany|22326    |ROUND SNACK BOXES SET OF4 WOODLAND |1233       |1   |
|Germany|15036    |ASSORTED COLOURS SILK FAN          |1164       |2   |
|Germany|POST     |POSTAGE                            |1114       |3   |
|Spain  |84997D   |CHILDRENS CUTLERY POLKADOT PINK    |729        |1   |
|Spain  |84997C   |CHILDRENS CUTLERY POLKADOT BLUE    |653        |2   |
|Spain  |20728    |LUNCH BAG CARS BLUE                |558        |3   |
+-------+---------+-----------------------------------+-----------+----+



---
### 📝 Bài tập 3: Khai phá giỏ hàng (Market Basket - collect_set)
**Yêu cầu**:
1. Gom nhóm theo `InvoiceNo`.
2. Dùng `collect_set("Description")` để lấy danh sách tên các mặt hàng trong hóa đơn.
3. Thêm cột `basket_size` đếm số lượng mặt hàng khác nhau trong giỏ bằng hàm `expr("size(items_in_basket)")`.
4. Lọc các hóa đơn có `basket_size >= 10` và hiển thị 3 hóa đơn đầu tiên.

In [23]:
# --- LỜI GIẢI BÀI TẬP 3 ---
solution_3 = clean_df \
    .groupBy("InvoiceNo") \
    .agg(collect_set("Description").alias("items_in_basket")) \
    .withColumn("basket_size", expr("size(items_in_basket)")) \
    .filter(col("basket_size") >= 10) \
    .orderBy(col("basket_size").desc())

print("Các giỏ hàng có trên 10 mặt hàng khác nhau:")
solution_3.show(3, truncate=50)

Các giỏ hàng có trên 10 mặt hàng khác nhau:
+---------+--------------------------------------------------+-----------+
|InvoiceNo|                                   items_in_basket|basket_size|
+---------+--------------------------------------------------+-----------+
|   573585|[MIDNIGHT BLUE PAIR HEART HAIR SLIDE, JUNGLE PO...|       1108|
|   581219|[MIDNIGHT BLUE PAIR HEART HAIR SLIDE, 3 RAFFIA ...|        749|
|   581492|[VICTORIAN SEWING BOX MEDIUM, POCKET BAG PINK P...|        731|
+---------+--------------------------------------------------+-----------+
only showing top 3 rows


---
### 📝 Bài tập 4: Phân nhóm khách hàng & Broadcast Join
**Yêu cầu**:
1. Tạo một DataFrame danh mục phân hạng khách hàng:
   - `VIP`: Khách hàng có `total_spent >= 5000`
   - `Gold`: Khách hàng có `2000 <= total_spent < 5000`
   - `Silver`: Khách hàng có `500 <= total_spent < 2000`
   - `Standard`: Các khách hàng còn lại.
2. Thêm cột `tier` vào DataFrame kết quả của Bài tập 1.
3. Tạo DataFrame quyền lợi ưu đãi:
   - `VIP`: Giảm giá 20%, Miễn phí vận chuyển.
   - `Gold`: Giảm giá 10%, Miễn phí vận chuyển.
   - `Silver`: Giảm giá 5%.
   - `Standard`: Tích điểm cơ bản.
4. Sử dụng `broadcast()` Join bảng ưu đãi này vào danh sách khách hàng.

In [24]:
# --- LỜI GIẢI BÀI TẬP 4 ---
# Gán nhãn tier
customer_tiered_df = solution_1.withColumn(
    "tier",
    when(col("total_spent") >= 5000, lit("VIP"))
    .when((col("total_spent") >= 2000) & (col("total_spent") < 5000), lit("Gold"))
    .when((col("total_spent") >= 500) & (col("total_spent") < 2000), lit("Silver"))
    .otherwise(lit("Standard"))
)

# Bảng chính sách ưu đãi
benefits_data = [
    ("VIP", "Giam 20% + Mien phi van chuyen toan quoc"),
    ("Gold", "Giam 10% + Mien phi van chuyen toan quoc"),
    ("Silver", "Giam 5% voucher thang"),
    ("Standard", "Tich luy 1% diem thuong")
]
benefits_df = spark.createDataFrame(benefits_data, ["tier", "benefit_policy"])

# Broadcast Left Join
enriched_customers = customer_tiered_df.join(
    broadcast(benefits_df),
    on="tier",
    how="left"
).select("CustomerID", "total_spent", "tier", "benefit_policy")

print("Danh sách khách hàng kèm chính sách ưu đãi phân hạng:")
enriched_customers.show(8, truncate=False)

Danh sách khách hàng kèm chính sách ưu đãi phân hạng:
+----------+------------------+--------+----------------------------------------+
|CustomerID|total_spent       |tier    |benefit_policy                          |
+----------+------------------+--------+----------------------------------------+
|17850     |5391.210000000009 |VIP     |Giam 20% + Mien phi van chuyen toan quoc|
|13047     |3237.54           |Gold    |Giam 10% + Mien phi van chuyen toan quoc|
|15311     |60767.899999999994|VIP     |Giam 20% + Mien phi van chuyen toan quoc|
|18074     |489.6             |Standard|Tich luy 1% diem thuong                 |
|17420     |598.8299999999999 |Silver  |Giam 5% voucher thang                   |
|16250     |389.44            |Standard|Tich luy 1% diem thuong                 |
|13408     |28117.039999999994|VIP     |Giam 20% + Mien phi van chuyen toan quoc|
|13767     |17220.359999999997|VIP     |Giam 20% + Mien phi van chuyen toan quoc|
+----------+------------------+--------+----

---
### 📝 Bài tập 5: Kiểm tra toàn vẹn dữ liệu danh mục với Left Anti Join
**Yêu cầu**:
1. Trích xuất danh sách tất cả các sản phẩm duy nhất xuất hiện trong hóa đơn (`clean_df.select("StockCode", "Description").distinct()`).
2. Giả lập một bảng danh mục chuẩn (Master Product Catalog) chỉ chứa một số mã hàng.
3. Sử dụng **`left_anti` Join** để tìm ra tất cả các mã sản phẩm có phát sinh bán trong giao dịch nhưng **chưa được đăng ký trong danh mục Master** (phục vụ bộ phận Data Quality Audit).

In [25]:
# --- LỜI GIẢI BÀI TẬP 5 ---
# Tạo bảng Master Catalog mẫu (chỉ gồm một số mã thông dụng)
master_catalog_data = [
    ("85123A", "Home Decor"),
    ("71053", "Lighting"),
    ("84406B", "Furnishing")
]
master_catalog_df = spark.createDataFrame(master_catalog_data, ["StockCode", "Category"])

# Trích xuất sản phẩm từ giao dịch
sold_products = clean_df.select("StockCode", "Description").distinct()

# Dùng Left Anti Join tìm các sản phẩm chưa có trong Master Catalog
missing_in_catalog = sold_products.join(
    master_catalog_df,
    on="StockCode",
    how="left_anti"
)

print(f"Số lượng sản phẩm phát sinh giao dịch chưa có trong Master Catalog: {missing_in_catalog.count():,}")
missing_in_catalog.show(5, truncate=False)

Số lượng sản phẩm phát sinh giao dịch chưa có trong Master Catalog: 4,156
+---------+---------------------------------+
|StockCode|Description                      |
+---------+---------------------------------+
|84879    |ASSORTED COLOUR BIRD ORNAMENT    |
|22745    |POPPY'S PLAYHOUSE BEDROOM        |
|22748    |POPPY'S PLAYHOUSE KITCHEN        |
|22749    |FELTCRAFT PRINCESS CHARLOTTE DOLL|
|22622    |BOX OF VINTAGE ALPHABET BLOCKS   |
+---------+---------------------------------+
only showing top 5 rows


---
## 9. Dọn dẹp tài nguyên & Giải phóng bộ nhớ

Khi hoàn thành công việc, ta giải phóng cache và đóng phiên làm việc của Spark.

In [26]:
# Giải phóng bộ nhớ Cache
clean_df.unpersist()

# Đóng SparkSession
spark.stop()
print("[*] Đã đóng SparkSession thành công. Tài nguyên hệ thống đã được giải phóng!")

[*] Đã đóng SparkSession thành công. Tài nguyên hệ thống đã được giải phóng!


---
## 10. Tổng kết

### 📌 Những kỹ năng cốt lõi đã đạt được:
1. **Làm sạch dữ liệu**: Kỹ thuật lọc và làm sạch trước khi tổng hợp giúp ngăn ngừa sai lệch số liệu kinh doanh.
2. **Aggregation**: Nắm vững các hàm thống kê từ cơ bản (`sum`, `avg`, `count`) đến nâng cao (`stddev`, `corr`) và kiểu phức tạp (`collect_set`).
3. **Window Functions**: Công cụ tối thượng để xếp hạng Top N theo nhóm và tính toán tổng lũy kế mà không làm mất cấu trúc chi tiết của từng dòng.
4. **Báo cáo đa chiều**: Khai thác sức mạnh của `rollup`, `cube` và `pivot` để tạo các báo cáo ma trận chuẩn BI.
5. **Làm chủ 8 kiểu Joins**: Hiểu rõ khi nào dùng Inner, Left, Full Outer và các loại Join đặc biệt như `left_semi` (tương đương `IN`) và `left_anti` (tương đương `NOT IN` để audit dữ liệu mồ côi).
6. **Tối ưu hóa Shuffle**: Nắm vững cơ chế **Broadcast Hash Join** với `broadcast()`, giải pháp sống còn để tối ưu hóa tốc độ truy vấn khi kết hợp bảng lớn với bảng danh mục.

### 🚀 Chuẩn bị cho Lecture 15&16:
Trong buổi học tiếp theo (**Lecture 15&16: Tối ưu hiệu năng Spark Job trong thực tế**), chúng ta sẽ khám phá:
- Cơ chế quản lý tài nguyên và phân vùng dữ liệu: Partitioning, Coalesce, Shuffle Spills.
- Định dạng lưu trữ tối ưu: Chuyển đổi từ CSV/JSON sang **Apache Parquet** và kỹ thuật nén Snappy.
- Kỹ thuật **Cache & Persistence** ở các Storage Level khác nhau (MEMORY_ONLY, MEMORY_AND_DISK).
- Đọc hiểu và phân tích **Spark Web UI**: DAG Visualization, Event Timeline, Stages và Tasks để tìm nguyên nhân gây chậm trễ (Bottleneck) của Pipeline.